# Season track map — one basin, one season

Plot **all storm tracks** for the 2005 North Atlantic season as `LineString`s, coloured by each storm's maximum Saffir-Simpson category, and annotate the strongest storm.

Uses `geometry='track'`, which returns one `LineString` per storm with summary attributes (`max_vmax_kt`, `min_mslp_hpa`, `max_category`, `ace`). Needs `pip install earthlens[tropycal]`.

## Setup

First the imports. `earthlens` provides the unified `EarthLens` entry point; `matplotlib` draws the track map at the end.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from earthlens.core import EarthLens

### Output directory

Tropycal is a vector backend, so `download()` writes the storm tracks to disk as well as returning them. Create the folder it writes into.

In [ ]:
OUT_DIR = Path('tropycal_output')
OUT_DIR.mkdir(exist_ok=True)

## All storm tracks for the 2005 North Atlantic season

### Build the request

We build the request first — the basin (`variables`), the season window, a bounding box over the North Atlantic, the `hurdat` best-track source, and `geometry='track'` so each storm comes back as a single `LineString` with summary attributes.

In [ ]:
season = EarthLens(
    variables=['north_atlantic'],
    data_source='tropycal',
    start='2005-06-01',
    end='2005-12-01',
    aoi=[-110.0, 0.0, -10.0, 60.0],
    source='hurdat',
    geometry='track',
    path=str(OUT_DIR),
)

### Download the tracks

`download()` is kept on its own line so the construct and the fetch read separately. The call is wrapped in a guard so the notebook degrades gracefully if the live best-track download is unavailable.

In [ ]:
tracks = None
try:
    tracks = season.download(progress_bar=False)
    print(len(tracks), 'storm tracks')
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

## Strongest storm of the season

Sorting the tracks by peak wind speed surfaces the season's strongest storm along with its minimum central pressure.

In [ ]:
if tracks is not None and len(tracks):
    strongest = tracks.sort_values('max_vmax_kt', ascending=False).iloc[0]
    print(
        strongest['name'],
        '- max wind',
        strongest['max_vmax_kt'],
        'kt, min pressure',
        strongest['min_mslp_hpa'],
        'hPa',
    )

## Map all tracks coloured by max category

Each track is drawn as a `LineString`, coloured by the storm's maximum Saffir-Simpson category, with the strongest storm labelled at its centroid.

In [ ]:
if tracks is not None and len(tracks):
    fig, ax = plt.subplots(figsize=(10, 6))
    tracks.plot(ax=ax, column='max_category', cmap='YlOrRd', legend=True, linewidth=1.5)
    strongest = tracks.sort_values('max_vmax_kt', ascending=False).iloc[0]
    cx, cy = strongest.geometry.centroid.x, strongest.geometry.centroid.y
    ax.annotate(strongest['name'], xy=(cx, cy), fontsize=11, fontweight='bold')
    ax.set_title('North Atlantic storm tracks, 2005 (coloured by max category)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.show()